In [1]:
import pandas as pd
import numpy as np

df_time = pd.DataFrame({'Tanggal': pd.date_range('2026-01-01', periods=3),
                        'Target': [139, 128, 172]})
df_time = df_time.set_index('Tanggal')

df_time['Time_Dummy'] = np.arange(len(df_time.index))
print(df_time)

            Target  Time_Dummy
Tanggal                       
2026-01-01     139           0
2026-01-02     128           1
2026-01-03     172           2


In [2]:
df_time['Lag_1'] = df_time['Target'].shift(1)
print(df_time)

            Target  Time_Dummy  Lag_1
Tanggal                              
2026-01-01     139           0    NaN
2026-01-02     128           1  139.0
2026-01-03     172           2  128.0


In [3]:
from sklearn.model_selection import train_test_split

df_clean = df_time.dropna()

X = df_clean[['Time_Dummy', 'Lag_1']]
y = df_clean['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print("X_train: \n", X_train, "\nY_train : \n", y_train)
print("X_test: \n", X_test, "\nY_test : \n", y_test)

X_train: 
             Time_Dummy  Lag_1
Tanggal                      
2026-01-02           1  139.0 
Y_train : 
 Tanggal
2026-01-02    128
Name: Target, dtype: int64
X_test: 
             Time_Dummy  Lag_1
Tanggal                      
2026-01-03           2  128.0 
Y_test : 
 Tanggal
2026-01-03    172
Name: Target, dtype: int64


Studi kasus

## 1. Load Data & Set Index

In [10]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load dataset
df = pd.read_csv('DailyDelhiClimateTest.csv')

# Convert date column to datetime and set as index
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')

print("=== Data Shape ===")
print(df.shape)
print("\n=== First 5 Rows ===")
print(df.head())
print("\n=== Data Info ===")
print(df.info())

=== Data Shape ===
(114, 4)

=== First 5 Rows ===
             meantemp   humidity  wind_speed  meanpressure
date                                                      
2017-01-01  15.913043  85.869565    2.743478     59.000000
2017-01-02  18.500000  77.222222    2.894444   1018.277778
2017-01-03  17.111111  81.888889    4.016667   1018.333333
2017-01-04  18.700000  70.050000    4.545000   1015.700000
2017-01-05  18.388889  74.944444    3.300000   1014.333333

=== Data Info ===
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 114 entries, 2017-01-01 to 2017-04-24
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   meantemp      114 non-null    float64
 1   humidity      114 non-null    float64
 2   wind_speed    114 non-null    float64
 3   meanpressure  114 non-null    float64
dtypes: float64(4)
memory usage: 4.5 KB
None


## 2. Feature Engineering

In [11]:
# Create Time_Dummy column (sequence from 0 to end)
df['Time_Dummy'] = np.arange(len(df.index))

# Create Lag_1 column (meantemp from previous day)
df['Lag_1'] = df['meantemp'].shift(1)

print("=== Dataset Setelah Feature Engineering ===")
print(df.head(10))

=== Dataset Setelah Feature Engineering ===
             meantemp   humidity  wind_speed  meanpressure  Time_Dummy  \
date                                                                     
2017-01-01  15.913043  85.869565    2.743478     59.000000           0   
2017-01-02  18.500000  77.222222    2.894444   1018.277778           1   
2017-01-03  17.111111  81.888889    4.016667   1018.333333           2   
2017-01-04  18.700000  70.050000    4.545000   1015.700000           3   
2017-01-05  18.388889  74.944444    3.300000   1014.333333           4   
2017-01-06  19.318182  79.318182    8.681818   1011.772727           5   
2017-01-07  14.708333  95.833333   10.041667   1011.375000           6   
2017-01-08  15.684211  83.526316    1.950000   1015.550000           7   
2017-01-09  14.571429  80.809524    6.542857   1015.952381           8   
2017-01-10  12.111111  71.944444    9.361111   1016.888889           9   

                Lag_1  
date                   
2017-01-01        N

## 3. Data Cleaning

In [12]:
# Remove rows with NaN values
print(f"Jumlah baris sebelum cleaning: {len(df)}")
df_clean = df.dropna()
print(f"Jumlah baris setelah cleaning: {len(df_clean)}")
print("\n=== Data Setelah Cleaning ===")
print(df_clean.head())

Jumlah baris sebelum cleaning: 114
Jumlah baris setelah cleaning: 113

=== Data Setelah Cleaning ===
             meantemp   humidity  wind_speed  meanpressure  Time_Dummy  \
date                                                                     
2017-01-02  18.500000  77.222222    2.894444   1018.277778           1   
2017-01-03  17.111111  81.888889    4.016667   1018.333333           2   
2017-01-04  18.700000  70.050000    4.545000   1015.700000           3   
2017-01-05  18.388889  74.944444    3.300000   1014.333333           4   
2017-01-06  19.318182  79.318182    8.681818   1011.772727           5   

                Lag_1  
date                   
2017-01-02  15.913043  
2017-01-03  18.500000  
2017-01-04  17.111111  
2017-01-05  18.700000  
2017-01-06  18.388889  


## 4. Data Splitting

In [13]:
# Define features (X) and target (y)
X = df_clean[['Time_Dummy', 'Lag_1']]
y = df_clean['meantemp']

# Split data: 80% Train, 20% Test (shuffle=False to preserve time order)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print("=== Data Splitting ===")
print(f"Total data: {len(df_clean)}")
print(f"Training data: {len(X_train)} ({len(X_train)/len(df_clean)*100:.1f}%)")
print(f"Testing data: {len(X_test)} ({len(X_test)/len(df_clean)*100:.1f}%)")
print("\n=== X_train (First 5) ===")
print(X_train.head())
print("\n=== X_test (First 5) ===")
print(X_test.head())

=== Data Splitting ===
Total data: 113
Training data: 90 (79.6%)
Testing data: 23 (20.4%)

=== X_train (First 5) ===
            Time_Dummy      Lag_1
date                             
2017-01-02           1  15.913043
2017-01-03           2  18.500000
2017-01-04           3  17.111111
2017-01-05           4  18.700000
2017-01-06           5  18.388889

=== X_test (First 5) ===
            Time_Dummy      Lag_1
date                             
2017-04-02          91  31.375000
2017-04-03          92  29.750000
2017-04-04          93  30.500000
2017-04-05          94  30.933333
2017-04-06          95  29.230769


## 5. Modelling - Linear Regression

In [14]:
# Build and train Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Display model coefficients
print("=== Model Linear Regression ===")
print(f"Intercept: {model.intercept_:.4f}")
print(f"Coefficient Time_Dummy: {model.coef_[0]:.4f}")
print(f"Coefficient Lag_1: {model.coef_[1]:.4f}")
print(f"\nFormula: meantemp = {model.intercept_:.4f} + {model.coef_[0]:.4f} * Time_Dummy + {model.coef_[1]:.4f} * Lag_1")

=== Model Linear Regression ===
Intercept: 2.1613
Coefficient Time_Dummy: 0.0318
Coefficient Lag_1: 0.8220

Formula: meantemp = 2.1613 + 0.0318 * Time_Dummy + 0.8220 * Lag_1


## 6. Prediction & Evaluation

In [15]:
# Make predictions on test data
y_pred = model.predict(X_test)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=== Hasil Evaluasi ===")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

=== Hasil Evaluasi ===
Mean Absolute Error (MAE): 0.9771
Root Mean Squared Error (RMSE): 1.2877


## 7. Output - Perbandingan Aktual vs Prediksi

In [16]:
# Create comparison table
hasil_df = pd.DataFrame({
    'Aktual': y_test.values,
    'Prediksi': np.round(y_pred, 2)
}, index=y_test.index)

# Add error column
hasil_df['Error'] = np.round(hasil_df['Aktual'] - hasil_df['Prediksi'], 2)
hasil_df['Abs_Error'] = np.round(np.abs(hasil_df['Error']), 2)

print("=== Tabel Perbandingan Aktual vs Prediksi ===")
print(hasil_df)
print("\n" + "="*50)
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print("="*50)

=== Tabel Perbandingan Aktual vs Prediksi ===
               Aktual  Prediksi  Error  Abs_Error
date                                             
2017-04-02  29.750000     30.84  -1.09       1.09
2017-04-03  30.500000     29.54   0.96       0.96
2017-04-04  30.933333     30.19   0.74       0.74
2017-04-05  29.230769     30.58  -1.35       1.35
2017-04-06  31.222222     29.21   2.01       2.01
2017-04-07  27.000000     30.88  -3.88       3.88
2017-04-08  25.625000     27.44  -1.82       1.82
2017-04-09  27.125000     26.34   0.79       0.79
2017-04-10  27.857143     27.60   0.26       0.26
2017-04-11  29.250000     28.24   1.01       1.01
2017-04-12  29.250000     29.41  -0.16       0.16
2017-04-13  29.666667     29.45   0.22       0.22
2017-04-14  30.500000     29.82   0.68       0.68
2017-04-15  31.222222     30.54   0.68       0.68
2017-04-16  31.000000     31.16  -0.16       0.16
2017-04-17  32.555556     31.01   1.55       1.55
2017-04-18  34.000000     32.32   1.68       1.68
2017